# Post-BMC TotalRisk reconstruction

This notebook repairs TotalRisk without changing the submitted notebooks. It treats the two-axis architecture as fixed, fits calibration on training outcomes only, freezes every learned quantity, and only then evaluates the temporal test cohort.

Required inputs are the scored train/test parquet artifacts consumed by notebook 04. They are not committed to this repository. Set `PHYSIORISK_TRAIN_PATH` and `PHYSIORISK_TEST_PATH`, or place them under `data/`. The input must contain `BaselineRisk` (DemoRisk), `PhysioRisk_2axis` (the submitted standardized score), and either `PhysioRisk_raw` or both fixed axes.

In [1]:
from pathlib import Path
import hashlib
import os
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
TABLE_DIR = ROOT / 'tables'
TABLE_DIR.mkdir(exist_ok=True)
TRAIN_PATH = Path(os.environ.get('PHYSIORISK_TRAIN_PATH', ROOT / 'data/physiorisk_2axis_v1_train_scored_from_frozen_baseline_levinenocrp.parquet'))
TEST_PATH = Path(os.environ.get('PHYSIORISK_TEST_PATH', ROOT / 'data/physiorisk_2axis_v1_test_scored_from_frozen_baseline_levinenocrp.parquet'))
TIME, EVENT, AGE = 'PERMTH_INT', 'MORTSTAT', 'RIDAGEYR'
AXIS_COEFS = {'HematologicStress': 0.433, 'RenalHepaticTurnover': 0.235}  # manuscript-rounded; use native raw column when available
CALIBRATION_PENALIZER = 0.0
EVALUATION_PENALIZER = 0.0
DDOF = 1

CANONICAL = {
    'train': {'path': TRAIN_PATH, 'sha256': 'e8f435b5a8b68dbb477f04bad280252f563b9323e9bc9dea4eb7e945c22cbb86', 'rows': 13090, 'deaths': 2520, 'cycles': {1999, 2001, 2003, 2005, 2007, 2009}},
    'test': {'path': TEST_PATH, 'sha256': '1455e24e8ca3e1f68eb58a779f7a4d48e46b8e54fa435ec00de5cf51256d08ca', 'rows': 6739, 'deaths': 416, 'cycles': {2011, 2013, 2015}},
}
REQUIRED_COLUMNS = {'SEQN', TIME, EVENT, AGE, 'RIAGENDR', 'cycle_start_year',
    'LBXSAL', 'LBXSCR', 'LBXGLU', 'LBXLYPCT', 'LBXMCV', 'LBXRDW', 'LBXSAPSI', 'LBXWBCSI',
    'BaselineRisk', 'HematologicStress', 'RenalHepaticTurnover', 'PhysioRisk_2axis', 'TotalRisk_2axis'}
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

missing_files = [str(spec['path']) for spec in CANONICAL.values() if not spec['path'].exists()]
if missing_files:
    raise FileNotFoundError('Missing non-committed canonical scored artifact(s): ' + ', '.join(missing_files))
for cohort, spec in CANONICAL.items():
    actual_hash = sha256_file(spec['path'])
    if actual_hash != spec['sha256']:
        raise AssertionError(f'{cohort} SHA256 mismatch: {actual_hash} != {spec["sha256"]}')
train = pd.read_parquet(TRAIN_PATH).copy()
test = pd.read_parquet(TEST_PATH).copy()
for cohort, df in [('train', train), ('test', test)]:
    spec = CANONICAL[cohort]
    missing = sorted(REQUIRED_COLUMNS - set(df.columns))
    if missing: raise AssertionError(f'{cohort} missing required columns: {missing}')
    if len(df) != spec['rows']: raise AssertionError(f'{cohort} row count {len(df)} != {spec["rows"]}')
    deaths = int(df[EVENT].sum())
    if deaths != spec['deaths']: raise AssertionError(f'{cohort} death count {deaths} != {spec["deaths"]}')
    cycles = set(df['cycle_start_year'].dropna().astype(int).unique())
    if cycles != spec['cycles']: raise AssertionError(f'{cohort} cycles {cycles} != {spec["cycles"]}')
    if df['SEQN'].isna().any() or not df['SEQN'].is_unique: raise AssertionError(f'{cohort} SEQN must be non-null and unique')
if not set(train['SEQN']).isdisjoint(set(test['SEQN'])):
    raise AssertionError('Training and test SEQN sets overlap')

In [2]:
# Preserve and validate the three distinct objects.
for df in (train, test):
    df['DemoRisk'] = df['BaselineRisk']
    df['PhysioRisk_z'] = df['PhysioRisk_2axis']

# Exact raw scale is used only when the canonical artifact contains it. Rounded manuscript
# coefficients are audited separately and are never silently promoted to the exact raw LP.
raw_exact_available = 'PhysioRisk_raw' in train.columns and 'PhysioRisk_raw' in test.columns
physio_mu = physio_sd = np.nan
rounded_raw_identity = {}
raw_identity_rows = []
rounded_mu = rounded_sd = np.nan
if raw_exact_available:
    physio_mu = float(train['PhysioRisk_raw'].mean())
    physio_sd = float(train['PhysioRisk_raw'].std(ddof=DDOF))
    for df in (train, test):
        df['PhysioRisk_z_from_raw'] = (df['PhysioRisk_raw'] - physio_mu) / physio_sd
        df['TotalRisk_raw'] = df['DemoRisk'] + df['PhysioRisk_raw']
else:
    axes = list(AXIS_COEFS)
    if all(c in train.columns and c in test.columns for c in axes):
        for name, df in [('train', train), ('test', test)]:
            df['PhysioRisk_raw_rounded_reconstruction'] = sum(AXIS_COEFS[c] * df[c] for c in axes)
        rounded_mu = float(train['PhysioRisk_raw_rounded_reconstruction'].mean())
        rounded_sd = float(train['PhysioRisk_raw_rounded_reconstruction'].std(ddof=DDOF))
        for name, df in [('train', train), ('test', test)]:
            z_rounded = (df['PhysioRisk_raw_rounded_reconstruction'] - rounded_mu) / rounded_sd
            error = z_rounded - df['PhysioRisk_z']
            rounded_raw_identity[name] = {'max_abs_error': float(error.abs().max()), 'mean_abs_error': float(error.abs().mean()), 'correlation': float(z_rounded.corr(df['PhysioRisk_z'])), 'allclose_1e-8': bool(np.allclose(z_rounded, df['PhysioRisk_z'], atol=1e-8, rtol=1e-7))}
            raw_identity_rows.append({'cohort': name, 'N': len(df), 'deaths': int(df[EVENT].sum()),
                'coef_HematologicStress_rounded': AXIS_COEFS['HematologicStress'], 'coef_RenalHepaticTurnover_rounded': AXIS_COEFS['RenalHepaticTurnover'],
                'rounded_raw_training_mean': rounded_mu, 'rounded_raw_training_sd_ddof_1': rounded_sd,
                **rounded_raw_identity[name], 'atol': 1e-8, 'rtol': 1e-7, 'exact_raw_available': False,
                'status': 'fails_historical_identity_rounded_coefficients'})
pd.DataFrame(raw_identity_rows).to_csv(TABLE_DIR / 'totalrisk_raw_identity_diagnostics.csv', index=False)

for df in (train, test):
    df['TotalRisk_original'] = df['DemoRisk'] + df['PhysioRisk_z']

# Historical identity checks are assertions, not refits.
if raw_exact_available:
    assert np.allclose(train['PhysioRisk_z_from_raw'], train['PhysioRisk_z'], atol=1e-8, rtol=1e-7)
    assert np.allclose(test['PhysioRisk_z_from_raw'], test['PhysioRisk_z'], atol=1e-8, rtol=1e-7)
if 'TotalRisk_2axis' in train:
    assert np.allclose(train['TotalRisk_original'], train['TotalRisk_2axis'], atol=1e-8, rtol=1e-7)
    assert np.allclose(test['TotalRisk_original'], test['TotalRisk_2axis'], atol=1e-8, rtol=1e-7)

In [3]:
# PRIMARY MODEL: all learned quantities below use training rows and training outcomes only.
fit_cols = [TIME, EVENT, 'DemoRisk', 'PhysioRisk_z']
cal = CoxPHFitter(penalizer=CALIBRATION_PENALIZER)
cal.fit(train[fit_cols].dropna(), duration_col=TIME, event_col=EVENT, batch_mode=True)
beta_demo = float(cal.params_['DemoRisk'])
beta_physio = float(cal.params_['PhysioRisk_z'])

# Prespecified offset sensitivity. lifelines has no Cox offset argument; fixing DemoRisk's coefficient
# requires a different package/custom likelihood, so this implementation records it as unavailable.
offset_status = 'not_fitted_lifelines_has_no_offset_api'

# Freeze coefficient scalars before any test-outcome access. Score calculation itself uses covariates only.
FROZEN = {'beta_Demo': beta_demo, 'beta_Physio': beta_physio, 'physio_train_mean': physio_mu, 'physio_train_sd': physio_sd}
for df in (train, test):
    df['CalibratedTotalRisk'] = FROZEN['beta_Demo'] * df['DemoRisk'] + FROZEN['beta_Physio'] * df['PhysioRisk_z']

coef_rows = []
for term in ['DemoRisk', 'PhysioRisk_z']:
    row = cal.summary.loc[term]
    coef_rows.append({'construction':'CalibratedTotalRisk','parameter':term.replace('Risk_z','Risk').replace('Risk','Risk'),
                      'estimate':float(row['coef']), 'standard_error':float(row['se(coef)']),
                      'ci_95_low':float(row['coef lower 95%']), 'ci_95_high':float(row['coef upper 95%']),
                      'penalizer':CALIBRATION_PENALIZER, 'log_likelihood':float(cal.log_likelihood_), 'fit_cohort':'training_only'})
coef_rows += [
 {'construction':'PhysioRisk_standardization','parameter':'training_mean_exact_raw','estimate':physio_mu,'standard_error':np.nan,'ci_95_low':np.nan,'ci_95_high':np.nan,'penalizer':np.nan,'log_likelihood':np.nan,'fit_cohort':'training_only' if raw_exact_available else 'unavailable_raw_scale'},
 {'construction':'PhysioRisk_standardization','parameter':'training_sd_ddof_1_exact_raw','estimate':physio_sd,'standard_error':np.nan,'ci_95_low':np.nan,'ci_95_high':np.nan,'penalizer':np.nan,'log_likelihood':np.nan,'fit_cohort':'training_only' if raw_exact_available else 'unavailable_raw_scale'},
 {'construction':'PhysioRisk_rounded_reconstruction','parameter':'training_mean_rounded_0.433_0.235','estimate':rounded_mu,'standard_error':np.nan,'ci_95_low':np.nan,'ci_95_high':np.nan,'penalizer':np.nan,'log_likelihood':np.nan,'fit_cohort':'training_covariates_only_approximation'},
 {'construction':'PhysioRisk_rounded_reconstruction','parameter':'training_sd_ddof_1_rounded_0.433_0.235','estimate':rounded_sd,'standard_error':np.nan,'ci_95_low':np.nan,'ci_95_high':np.nan,'penalizer':np.nan,'log_likelihood':np.nan,'fit_cohort':'training_covariates_only_approximation'},
 {'construction':'DemoRisk_offset_sensitivity','parameter':'status','estimate':np.nan,'standard_error':np.nan,'ci_95_low':np.nan,'ci_95_high':np.nan,'penalizer':np.nan,'log_likelihood':np.nan,'fit_cohort':offset_status}]
pd.DataFrame(coef_rows).to_csv(TABLE_DIR / 'totalrisk_training_coefficients.csv', index=False)

In [4]:
# Comparator reconstruction, matching notebook 04. No test fitting.
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from patsy import dmatrix, build_design_matrices
pheno_cols = ['LBXSAL','LBXSCR','LBXGLU','LBXMCV','LBXRDW','LBXWBCSI','LBXLYPCT','LBXSAPSI']
imp = SimpleImputer(strategy='median')
train[pheno_cols] = imp.fit_transform(train[pheno_cols]); test[pheno_cols] = imp.transform(test[pheno_cols])
def add_pheno(df):
    df['PhenoAge_NCP'] = (-19.907 - .0336*df.LBXSAL + .0095*df.LBXSCR + .1953*np.log(df.LBXGLU.clip(lower=1e-6))
      - .0120*df.LBXLYPCT + .0268*df.LBXMCV + .3306*df.LBXRDW + .0019*df.LBXSAPSI + .0554*df.LBXWBCSI + .0804*df[AGE])
add_pheno(train); add_pheno(test)
Xtr = dmatrix(f'cr({AGE}, df=4) + C(RIAGENDR)', train, return_type='dataframe')
Xte = build_design_matrices([Xtr.design_info], test, return_type='dataframe')[0]
accel = LinearRegression().fit(Xtr, train['PhenoAge_NCP'])
train['PhenoAge_NCP_acceleration'] = train['PhenoAge_NCP'] - accel.predict(Xtr)
test['PhenoAge_NCP_acceleration'] = test['PhenoAge_NCP'] - accel.predict(Xte)

def evaluate_actual_sd(df, label, col):
    d = df[[TIME, EVENT, AGE, col]].dropna().copy()
    actual_sd = float(d[col].std(ddof=1))
    d['predictor_z_for_hr'] = (d[col] - d[col].mean()) / actual_sd
    assert np.isclose(d['predictor_z_for_hr'].std(ddof=1), 1.0)
    cph = CoxPHFitter(penalizer=EVALUATION_PENALIZER).fit(
        d[[TIME,EVENT,'predictor_z_for_hr']], TIME, EVENT, batch_mode=True,
        fit_options={'step_size': 0.5, 'max_steps': 1000, 'precision': 1e-9})
    s = cph.summary.loc['predictor_z_for_hr']
    return {'model':label, 'N':len(d), 'deaths':int(d[EVENT].sum()),
      'C_index':concordance_index(d[TIME], -d[col], d[EVENT]), 'actual_test_SD':actual_sd,
      'HR_per_actual_1SD':float(s['exp(coef)']), 'HR_95CI_low':float(s['exp(coef) lower 95%']),
      'HR_95CI_high':float(s['exp(coef) upper 95%']), 'correlation_with_age':float(d[col].corr(d[AGE])),
      'HR_standardization_verified':True, 'HR_evaluation_penalizer':EVALUATION_PENALIZER}
scores = [('DemoRisk','DemoRisk'),('PhysioRisk_convergence_corrected','PhysioRisk_z'),('TotalRisk_original','TotalRisk_original'),
          ('CalibratedTotalRisk','CalibratedTotalRisk'),
          ('PhenoAge NCP','PhenoAge_NCP'),('PhenoAge NCP acceleration','PhenoAge_NCP_acceleration')]
comparison = pd.DataFrame(evaluate_actual_sd(test, label, col) for label, col in scores)
submitted_physio = {'model':'PhysioRisk_submitted', 'N':6739, 'deaths':416, 'C_index':0.6541097324576873,
    'actual_test_SD':float(test['PhysioRisk_z'].std(ddof=1)), 'HR_per_actual_1SD':2.7716579077104098,
    'HR_95CI_low':2.655680584394424, 'HR_95CI_high':2.8927001245992834,
    'correlation_with_age':0.10623345250003381, 'HR_standardization_verified':True,
    'HR_evaluation_penalizer':0.0, 'status':'submitted_table2_nonconverged_fit_preserved'}
comparison = pd.concat([comparison.iloc[:1], pd.DataFrame([submitted_physio]), comparison.iloc[1:]], ignore_index=True)
if raw_exact_available:
    comparison = pd.concat([comparison, pd.DataFrame([evaluate_actual_sd(test, 'TotalRisk_raw', 'TotalRisk_raw')])], ignore_index=True)
else:
    comparison = pd.concat([comparison, pd.DataFrame([{'model':'TotalRisk_raw','N':len(test),'deaths':int(test[EVENT].sum()),'HR_standardization_verified':False,'status':'not_computable_exact_raw_scale_unavailable'}])], ignore_index=True)
comparison.to_csv(TABLE_DIR / 'totalrisk_reconstruction_comparison.csv', index=False)
comparison

,model,N,deaths,C_index,actual_test_SD,HR_per_actual_1SD,HR_95CI_low,HR_95CI_high,correlation_with_age,HR_standardization_verified,HR_evaluation_penalizer,status
0,DemoRisk,6739,416,0.826876,1.308431,3.735674,3.346769,4.169771,0.969734,True,0.0,NaN
1,PhysioRisk_submitted,6739,416,0.654110,0.949861,2.771658,2.655681,2.892700,0.106233,True,0.0,submitted_table2_nonconverged_fit_preserved
2,PhysioRisk_convergence_corrected,6739,416,0.654110,0.949861,1.387468,1.337516,1.439286,0.106233,True,0.0,NaN
3,TotalRisk_original,6739,416,0.839760,1.689100,1.877466,1.815234,1.941831,0.810927,True,0.0,NaN
4,CalibratedTotalRisk,6739,416,0.843153,1.457188,3.259002,3.035997,3.498387,0.946491,True,0.0,NaN
5,PhenoAge NCP,6739,416,0.841108,1.666348,3.976116,3.636923,4.346944,0.961369,True,0.0,NaN
6,PhenoAge NCP acceleration,6739,416,0.637857,0.458601,1.432147,1.359932,1.508198,0.033299,True,0.0,NaN
7,TotalRisk_raw,6739,416,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,not_computable_exact_raw_scale_unavailable
